# Parte 4 — Reflexión

**Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña**

Las cifras citadas salen de `resultados/parte2_k3.csv` y `resultados/parte2_k5.csv`; la celda siguiente las recalcula para que el texto se pueda verificar.

In [1]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd()
while raiz != raiz.parent and not (raiz / "pyproject.toml").exists():
    raiz = raiz.parent

for k in (3, 5):
    df = pd.read_csv(raiz / "resultados" / f"parte2_k{k}.csv")
    resp, neg = df[df["respondible"]], df[~df["respondible"]]
    aciertos = int(resp["hit"].astype(bool).sum())
    print(f"k={k}: aciertos={aciertos}/{len(resp)} respondibles · Hit Rate={aciertos/len(resp):.3f} · "
          f"con negativas en el denominador={aciertos/len(df):.3f} · techo con negativas={len(resp)/len(df):.2f} · "
          f"MRR={resp['reciprocal_rank'].mean():.3f}")
    print(f"      abstención correcta={neg['abstuvo'].astype(bool).mean():.3f} · indebida={resp['abstuvo'].astype(bool).mean():.3f} · "
          f"score_top1 negativas={neg['score_top1'].round(3).tolist()} · rango respondibles="
          f"[{resp['score_top1'].min():.3f}, {resp['score_top1'].max():.3f}]")

k=3: aciertos=5/8 respondibles · Hit Rate=0.625 · con negativas en el denominador=0.500 · techo con negativas=0.80 · MRR=0.542
      abstención correcta=1.000 · indebida=0.125 · score_top1 negativas=[0.567, 0.577] · rango respondibles=[0.554, 0.699]
k=5: aciertos=5/8 respondibles · Hit Rate=0.625 · con negativas en el denominador=0.500 · techo con negativas=0.80 · MRR=0.542
      abstención correcta=1.000 · indebida=0.250 · score_top1 negativas=[0.567, 0.577] · rango respondibles=[0.554, 0.699]


## 1. El techo de 0,70, con nuestras cifras

Con la plantilla anterior, el Hit Rate dividía los aciertos entre las diez preguntas, incluidas las negativas, que no tienen nada que recuperar y cuentan siempre como fallo. Con nuestro golden set (8 respondibles, 2 negativas), los 5 aciertos habrían dado 5/10 = 0,50 en lugar de 0,625, y un sistema perfecto no pasaría de 0,80. Ese número mezclaba dos cosas: cuánto recupera el sistema y cuántas preguntas sin respuesta tiene el set. Al sacar las negativas del denominador, el Hit Rate (0,625) y el MRR (0,542) miden solo la recuperación. Para las negativas, la métrica correcta es la abstención, porque el índice no se queja: devuelve vecinos con puntajes de 0,567 y 0,577, dentro del rango de las respondibles. Se mide al generar: abstención correcta 1,0 y, como contrapeso, indebida 0,125 (k=3) y 0,25 (k=5). Las dos tasas se leen juntas.

## 2. Una pregunta global y GraphRAG

**Pregunta global:** «¿Cómo evolucionan, de 2017 a 2023, las técnicas para alinear modelos de lenguaje con lo que quieren las personas, y qué tienen en común?»

Ningún fragmento contiene esa respuesta: exige recorrer los seis papers y sintetizar. Nuestro RAG entrega 5 fragmentos de 512 tokens, menos del 1 % de los 530, elegidos por parecido con la pregunta y no por cobertura. El caso 2 lo mostró: una pregunta sobre "alineamiento" trae solo fragmentos de InstructGPT, así que la respuesta ignoraría DPO y Constitutional AI y parecería completa sin serlo. Edge et al. (2024) distinguen estas preguntas globales de las que un fragmento puede contestar. GraphRAG, al indexar, usa un LLM para extraer entidades y relaciones (RLHF, modelo de recompensa, PPO, DPO, retroalimentación de IA), agrupa el grafo en comunidades y pre-genera un resumen de cada una. Ante la pregunta global responde en map-reduce sobre todos esos resúmenes, cubriendo el corpus completo, a cambio de un indexado mucho más costoso.